# Cluster Then Classify — Phase 5
### The reveal

Every decision so far was made blind. This notebook opens the true labels for
the first time, and that ordering is the only reason any number below means
anything.

**The headline result is not accuracy. It is accuracy relative to a model
trained on real labels** — the *% of ceiling* column. A bare accuracy figure
invites the reader to compare it to whatever benchmark they have in mind; the
ratio answers the question they actually care about, which is how much you gave
up by not annotating.

In [ ]:
import sys, json
sys.path.insert(0, '..'); sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config as C
import evaluate as EV
from classify import make_split

pd.set_option('display.max_colwidth', 120)

In [ ]:
boot = pd.read_csv(C.P_BOOTSTRAPPED)
emb = np.load(C.P_EMB)
labels = np.load(C.P_CLUSTER_LABELS)

# ---- the moment of truth --------------------------------------------------
y_true_idx = pd.read_csv(C.P_QUARANTINE)[C.LABEL_COL].values
y_true = pd.Series(y_true_idx).map(C.CLASS_NAMES).values

assert len(boot) == len(emb) == len(labels) == len(y_true), 'row counts drifted'
print(f'{len(boot)} rows, {len(set(labels))} clusters, {len(set(y_true))} true classes')
pd.Series(y_true).value_counts()

## 5.1 Did the clustering recover the structure?

ARI and NMI are computed on the partition alone and never look at cluster
*names*, so they answer "did this find the right groups" independently of
whether the LLM named them well.

In [ ]:
quality = EV.clustering_quality(labels, y_true)
print(f"ARI {quality['ARI']:.3f}   NMI {quality['NMI']:.3f}")

# ARI of 0 means the partition is no better than chance; 1 is exact recovery.
# Anything above ~0.4 on four-class text is real structure, not noise.

## 5.2 Align the clusters to the true classes

Clusters are unordered and the LLM's names are free-form, so they must be
matched to true classes before accuracy exists at all. Two rules:

- **Use the Hungarian algorithm, not your eyes.** Eyeballing the contingency
  matrix means picking the assignment that flatters the result.
- **Never match on name similarity.** If the LLM happened to output "Sports"
  and you matched it to the true class "Sports" by string equality, you would
  be rewarding it for guessing which public dataset this is — not for
  understanding the documents.

In [ ]:
mapping, cluster_acc, cm = EV.align(labels, y_true)

print('cluster -> class mapping (Hungarian):')
for k, v in sorted(mapping.items(), key=lambda kv: str(kv[0])):
    print(f'  cluster {k} -> {v}')
print(f'\ncluster alignment accuracy: {cluster_acc:.1%}')
cm

In [ ]:
# Accuracy of the bootstrapped labels themselves, before any classifier.
# This is the ceiling on what the downstream model can learn from them.
_, boot_label_acc, boot_cm = EV.align(boot['llm_label'].values, y_true)
print(f'bootstrapped label accuracy: {boot_label_acc:.1%}')
boot_cm

## 5.3 The ablation table

Every condition trains on a different label source and is scored against the
**same** true test labels, so the rows are directly comparable.

A note on how the mapping is fit: the cluster→class map used to score the
bootstrapped conditions is derived from the **training split only**. Fitting it
on all rows would let test-set ground truth influence the mapping, which is
leakage — and it inflates precisely the conditions you are trying to defend.

In [ ]:
y_boot = boot['llm_label'].values
tr, te = make_split(len(boot), y_boot)

results, preds, boot_map = EV.run_ablations(
    boot['text'].values, emb, y_boot, y_true, tr, te
)
table = EV.ablation_table(results)

show = table.copy()
show['accuracy'] = show['accuracy'].map('{:.3f}'.format)
show['% of ceiling'] = show['% of ceiling'].map('{:.1%}'.format)
show

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
t = table.sort_values('accuracy')
colors = ['#888' if 'Bootstrapped' not in c else '#2b6cb0' for c in t['condition']]
ax.barh(t['condition'], t['accuracy'], color=colors)
ax.axvline(results['ceiling_tfidf'], ls='--', c='k', lw=1, label='ceiling')
ax.set_xlabel('accuracy against true labels'); ax.legend(); ax.set_xlim(0, 1)
for i, v in enumerate(t['accuracy']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

### Reading the table

Two rows decide whether the project worked.

**Small supervised.** This is the row a sceptical reader reaches for first. If
hand-labelling 100 examples beats the entire pipeline, the pipeline is not worth
building, and you must say so plainly rather than bury it.

**Hybrid.** If adding 100 true labels barely moves the number, the bottleneck is
not annotation volume — it is the cluster boundaries themselves, and more human
labels will not fix it. If it jumps, you have a cheap and honest improvement to
recommend.

In [ ]:
from sklearn.metrics import classification_report

pred_mapped = np.array([boot_map.get(p, p) for p in preds['bootstrapped_tfidf']])
print(classification_report(y_true[te], pred_mapped, zero_division=0))

## 5.4 Error analysis

Merging and splitting have different causes and different fixes, and describing
them correctly is what makes a writeup read as analysis rather than a report
card.

- **Merged** — one cluster holds two true classes. The embedding does not
  separate them. More clusters may help; more human labels will not.
- **Split** — one true class spread over several clusters. Usually harmless
  under many-to-one mapping, and often means the class genuinely contains
  sub-topics.

In [ ]:
errors = EV.error_analysis(labels, y_true_idx)
print(errors['contingency'].to_string())

print()
for m in errors['merged']:
    print(f"MERGED : cluster {m['cluster']} mixes {m['classes']} in shares {m['shares']}")
for s in errors['split']:
    print(f"SPLIT  : class {s['class']} spread over clusters {s['clusters']} {s['shares']}")
if not errors['merged'] and not errors['split']:
    print('clean one-to-one recovery')

In [ ]:
# Read the documents the pipeline got wrong. Numbers tell you how much;
# only the text tells you why.
wrong = pd.DataFrame({
    'text': boot['text'].values[te],
    'predicted': pred_mapped,
    'true': y_true[te],
})
wrong = wrong[wrong['predicted'] != wrong['true']]
print(f'{len(wrong)} errors out of {len(te)}\n')
for r in wrong.head(8).itertuples():
    print(f'  [{r.true} -> {r.predicted}]  {r.text[:120]}')

## 5.5 Was it worth it?

The cost figures are rough by construction. The order of magnitude is the
finding; do not present the second decimal as if it were measured.

In [ ]:
N_API_CALLS = 1        # 1 labelling call, + ~15 more if you ran the spot check
cost = EV.cost_comparison(len(boot), N_API_CALLS)

print(f"API spend      : ${cost['api_cost_usd']}")
print(f"Human estimate : ${cost['human_cost_usd']}  ({cost['human_hours']}h at 200 docs/h, $25/h)")
print(f"Ratio          : {cost['ratio']:,.0f}x")

recovery = results['bootstrapped_tfidf'] / results['ceiling_tfidf']
print(f"\nRecovered {recovery:.1%} of supervised performance with zero human annotation.")

In [ ]:
json.dump({
    'clustering_quality': quality,
    'cluster_alignment_accuracy': cluster_acc,
    'bootstrapped_label_accuracy': boot_label_acc,
    'ablation': table.to_dict('records'),
    'label_map': {str(k): str(v) for k, v in boot_map.items()},
    'cost': cost,
}, open(C.P_RESULTS, 'w'), indent=2)

table.to_csv(C.P_ABLATION, index=False)
print(f'wrote {C.P_RESULTS.name} and {C.P_ABLATION.name}')

---
## Writing it up

Lead with the ratio, not the accuracy:

> On AG News, clustering plus LLM labelling recovered **X% of the accuracy of a
> fully supervised model using zero human annotations**, at an API cost of under
> a cent against an estimated $N of manual labelling. Adding 100 hand-labelled
> examples changed the result by Y points.

Then include, in this order: the ablation table, the UMAP plot, the merged/split
findings, and a section on what failed.

**Write the failure section.** A project reporting only wins reads as
unexamined. Naming the classes the pipeline could not separate, and explaining
why they are genuinely hard to separate, is what makes everything else
credible.

### Things worth trying if you have time left

- **More clusters than classes.** Over-clustering and letting several clusters
  map to one class often beats setting k to the number of classes, because it
  lets the model carve a broad class into the sub-topics that actually cluster.
  In the offline dry run on this corpus, k=8 recovered ~92% of ceiling against
  ~82% at k=4. Worth testing properly.
- **A better embedding.** `all-mpnet-base-v2` over MiniLM.
- **Confidence-weighted training.** Down-weight documents far from their
  centroid; they are the ones propagation most likely mislabelled.